1. create included sample list

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
# Set these to match config/paths.yaml
import os
PHENO_DIR       = "<set to cfg.gwas.pheno_dir in config/paths.yaml>"
COVAR_DIR       = "<set to cfg.gwas.covar_dir in config/paths.yaml>"
KINSHIP_TABLE   = "<set to cfg.ukb.kinship_table in config/paths.yaml>"
WITHDRAWAL_LIST = "<set to cfg.ukb.withdrawal_list in config/paths.yaml>"
MASTER_CSV      = "<set to cfg.ukb.master_csv in config/paths.yaml>"
QCTOOL          = "<set to cfg.tools.qctool in config/paths.yaml>"
BGENIX          = "<set to cfg.tools.bgenix in config/paths.yaml>"
BGEN            = "<set to cfg.ukb.bgen in config/paths.yaml>"          # final all_filtered MRI bgen
BGEN_SAMPLE     = "<set to cfg.ukb.bgen_sample in config/paths.yaml>"   # final MRI bgen .sample
# EXTERNAL inputs/outputs (not in config/paths.yaml):
MRI_SAMPLE_LIST   = "<EXTERNAL: MRI-cohort sample-inclusion list (ID_1)>"
GMMAT_SUBJECT_DIR = "<EXTERNAL: directory of per-cohort GMMAT subject-ID lists>"
RAW_BGEN_DIR      = "<EXTERNAL: UK Biobank imputed per-chromosome BGEN directory>"
RAW_BGEN_SAMPLE   = "<EXTERNAL: UK Biobank imputed .sample file>"
MRI_BGEN_DIR      = "<EXTERNAL: working directory for the MRI-subset BGEN files>"
GMMAT_BGEN_GLOB   = "<EXTERNAL: glob for GMMAT subset per-chromosome BGEN files>"
SNP_LIST_SUMSTATS = "<EXTERNAL: reference sumstats file providing the SNP inclusion list>"


In [ ]:
import pandas as pd

In [ ]:
T1_pheno = pd.read_table(os.path.join(PHENO_DIR, "T1_pheno_discovery"), sep=' ')
T2_pheno = pd.read_table(os.path.join(PHENO_DIR, "T2_pheno_discovery"), sep=' ')
T1_covar = pd.read_table(os.path.join(COVAR_DIR, "T1_covar_discovery"), sep=' ')
T2_covar = pd.read_table(os.path.join(COVAR_DIR, "T2_covar_discovery"), sep=' ')

In [ ]:
T1_pheno_r = pd.read_table(os.path.join(PHENO_DIR, "T1_pheno_replication"), sep=' ')
T2_pheno_r = pd.read_table(os.path.join(PHENO_DIR, "T2_pheno_replication"), sep=' ')
T1_covar_r = pd.read_table(os.path.join(COVAR_DIR, "T1_covar_replication"), sep=' ')
T2_covar_r = pd.read_table(os.path.join(COVAR_DIR, "T2_covar_replication"), sep=' ')

In [ ]:
withdrawal = pd.read_table(WITHDRAWAL_LIST, header=None)

In [ ]:
discovery_iid = (set(T1_pheno.IID).intersection(T1_covar.IID)).union(set(T2_pheno.IID).intersection(T2_covar.IID)).difference(withdrawal[0])

In [ ]:
replication_iid = (set(T1_pheno_r.IID).intersection(T1_covar_r.IID)).union(set(T2_pheno_r.IID).intersection(T2_covar_r.IID)).difference(withdrawal[0])

In [ ]:
kinship = pd.read_table(KINSHIP_TABLE, sep=' ')[['ID1', 'ID2', 'Kinship']]

In [ ]:
kinship = kinship[kinship.Kinship != -1]

In [ ]:
replication_exclusion = []
for id1, id2, _ in kinship.values:
    if id1 in discovery_iid and id2 in replication_iid:
        replication_exclusion.append(id2)
    elif id1 in replication_iid and id2 in discovery_iid:
        replication_exclusion.append(id1)

In [ ]:
len(replication_exclusion)

In [ ]:
replication_iid = replication_iid.difference(replication_exclusion)

In [ ]:
sample_iid = discovery_iid.union(replication_iid)

In [ ]:
with open(MRI_SAMPLE_LIST, 'w') as f:
    f.write("ID_1\n")
    f.writelines(map(lambda x: str(int(x))+'\n', sample_iid))

2. check for sex mismatch

In [ ]:
from dask import dataframe as dd
from multiprocessing import Pool

In [ ]:
sample_iid = pd.read_table(MRI_SAMPLE_LIST)

In [ ]:
df = dd.read_csv(MASTER_CSV, dtype='object')

In [ ]:
df['eid'] = df.eid.astype('i')

In [ ]:
df = df.loc[df.eid.isin(sample_iid[0]), ["eid", "31-0.0", "22001-0.0"]].compute(scheduler="processes")

In [ ]:
sex_mismatch = df[df["31-0.0"] != df["22001-0.0"]].eid

In [ ]:
sample_iid = set(sample_iid[0]).difference(sex_mismatch)

In [ ]:
with open(MRI_SAMPLE_LIST, 'w') as f:
    f.write("ID_1\n")
    f.writelines(map(lambda x: str(int(x))+'\n', sample_iid[0]))

In [ ]:
gmmat_subject = set()
for i in [1, 2]:
    for j in ['discovery', 'replication']:
        with open(os.path.join(GMMAT_SUBJECT_DIR, f"T{i}_{j}_subject.txt"), 'r') as f:
            gmmat_subject.update(list(map(int, f.read().strip().split('\n'))))

In [ ]:
with open("GMMAT_samples.txt", 'w') as f:
    f.write("ID_1\n")
    f.writelines(map(lambda x: str(int(x))+'\n', gmmat_subject))

3. combine bgen files from all autosomes 

In [ ]:
from glob import glob
from multiprocessing import Pool
import os

In [ ]:
qctool_path = QCTOOL

In [ ]:
bgens = glob(os.path.join(RAW_BGEN_DIR, "*.bgen"))

In [ ]:
sample = RAW_BGEN_SAMPLE

In [ ]:
def proc(x):
    c = x.split("_")[-2]
    s = f"-g {x} -s {sample}"
    cmd = f"nohup {qctool_path} {s} -incl-samples {MRI_SAMPLE_LIST} -og {os.path.join(MRI_BGEN_DIR, f'MRI_samples_{c}.bgen')} -os {os.path.join(MRI_BGEN_DIR, f'MRI_samples_{c}.sample')} > {c}.out"
    os.system(cmd)

In [ ]:
with Pool(len(bgens)) as p:
    p.map(proc, bgens)

create index using bgenix

In [ ]:
bgenix_path = BGENIX

In [ ]:
bgens_subset = glob(GMMAT_BGEN_GLOB)

In [ ]:
cmd = f"{bgenix_path} -g {bgens_subset[0]} -index"

In [ ]:
print(cmd)

In [ ]:
# ./cat-bgen -g {MRI_BGEN_DIR}/*.bgen -og {MRI_BGEN_DIR}/all.bgen

In [ ]:
cmd4 = f"{qctool_path} -g {os.path.join(MRI_BGEN_DIR, 'all.bgen')} -snp-stats -osnp {os.path.join(MRI_BGEN_DIR, 'snp-stats.txt')}"

In [ ]:
print(cmd4)
# too slow

In [ ]:
snp_list = pd.read_table(SNP_LIST_SUMSTATS, compression="gzip")

In [ ]:
snp_list.SNP.to_csv("incl_rsid", index=False)

In [ ]:
bgens = glob(os.path.join(MRI_BGEN_DIR, "*chr*.bgen"))

In [ ]:
def filter_snp(x):
    x_name = x.split('/')[-1].split('.')[0]
    os.system(f"{qctool_path} -g {x} -og {os.path.join(MRI_BGEN_DIR, f'{x_name}_filtered.bgen')} -incl-rsids incl_rsid")

In [ ]:
with Pool(len(bgens)) as p:
    p.map(filter_snp, bgens)

fastGWA